# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary.

# Used Car Valuation & Price Driver Analysis (WIP)

## Phase 1: Business Understanding

My client is a used car dealership looking to optimize its inventory acquisition and pricing strategy. In this analysis, my goal is to identify what features make a vehicle more or less expensive to consumers.

By applying regression modeling to historical vehicle marketplace data, I am quantifying consumer valuation drivers to provide actionable recommendations for the dealership:
* **Trade-in Appraisals:** Establishing accurate baseline offers based on vehicle age, mileage, and condition.
* **Auction Sourcing:** Identifying high-value attributes (such as drivetrains, engine sizes, and vehicle classes) that command market premiums.
* **Title & Risk Penalties:** Measuring the financial depreciation associated with salvage and rebuilt titles.

### Project Roadmap (CRISP-DM Framework)
1. **Business Understanding:** Framing dealership valuation objectives.
2. **Data Understanding & Cleanup:** Profiling raw schema, inspecting missing values, and executing domain-driven imputation.
3. **Data Preparation & Feature Engineering:** Analyzing target correlations, filtering market outliers, applying transformations, and encoding categoricals.
4. **Modeling:** Training and tuning Linear, Ridge, and Lasso regression models.
5. **Evaluation:** Assessing performance metrics ($R^2$, RMSE, MAE) and residual behavior.
6. **Deployment & Dealership Recommendations:** Translating model weights into practical inventory strategies.


In [ ]:
# for work in Google CoLab only
from google.colab import drive
drive.mount('/content/drive')
# replace path accordingly
path = 'drive/MyDrive/all_data/11.1/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import Lasso, LinearRegression, Ridge  # Regression models
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split  # Evaluation splitting
from sklearn.preprocessing import StandardScaler  # Feature scaling

In [ ]:
vehicles_df = pd.read_csv(path + 'data/vehicles.csv')
vehicles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   region        426880 non-null  object 
 2   price         426880 non-null  int64  
 3   year          425675 non-null  float64
 4   manufacturer  409234 non-null  object 
 5   model         421603 non-null  object 
 6   condition     252776 non-null  object 
 7   cylinders     249202 non-null  object 
 8   fuel          423867 non-null  object 
 9   odometer      422480 non-null  float64
 10  title_status  418638 non-null  object 
 11  transmission  424324 non-null  object 
 12  VIN           265838 non-null  object 
 13  drive         296313 non-null  object 
 14  size          120519 non-null  object 
 15  type          334022 non-null  object 
 16  paint_color   296677 non-null  object 
 17  state         426880 non-null  object 
dtypes: f

# Exploratory Data Analysis (EDA)
In this section, we explore the structure, completeness, and initial distributions of the `vehicles_df` dataset to guide our cleaning decisions.

## Phase 2: Data Understanding and Initial Cleanup

In this phase, I explore the raw dataset and examine feature distributions. By understanding what the data represents, I combine industry conventions with my domain background to assess data quality, evaluate missing data patterns, and clean the vehicle inventory.

I achieve this by:
* Performing foundational structural cleaning to prune noisy and redundant attributes.
* Filtering non-viable records to isolate dealership-grade inventory.
* Resolving missing specifications through hierarchical domain imputation.

This establishes a clean, robust baseline for the next phases: Exploratory Data Analysis (EDA) and Feature Engineering.


**Note**: I have created a helper function to output a clean-formatted dump of the NaN values so I can check it repeatedly.

In [ ]:
def get_nan_df(df: pd.DataFrame) -> pd.DataFrame:
  """Returns a numeric DataFrame of missing counts and percentages for all columns."""
  nan_counts = df.isna().sum()
  nan_pct = (nan_counts / len(df) * 100).round(1)

  return pd.DataFrame({'Missing Count': nan_counts, 'Percentage (%)': nan_pct})


def get_nan_summary(
    df: pd.DataFrame, df_name: str = 'DataFrame'
) -> pd.DataFrame:
  """Computes, prints, and returns a sorted summary table of missing values in a DataFrame.

  Args:
      df: The input pandas DataFrame to analyze.
      df_name: Optional name for labeling the output message.

  Returns:
      A sorted pandas DataFrame summarizing problematic columns with missing
      values, including a formatted TOTAL row. Returns an empty DataFrame if no
      missing values exist.
  """
  nan_table = get_nan_df(df)

  # 2. Filter to only columns with missing data and sort descending
  problematic_table = (
      nan_table[nan_table['Missing Count'] > 0]
      .sort_values(by='Missing Count', ascending=False)
      .copy()
  )

  if problematic_table.empty:
    print(f"<{df_name}>: No missing values detected.\n")
    return problematic_table

  # 3. Calculate and append the summary total row
  total_missing = problematic_table['Missing Count'].sum()
  total_pct = problematic_table['Percentage (%)'].sum().round(1)

  problematic_table.loc['TOTAL'] = [total_missing, total_pct]
  problematic_table['Missing Count'] = problematic_table[
      'Missing Count'
  ].astype(int)

  print(f"<{df_name}> Missing Data Breakdown\n")
  return problematic_table

print(get_nan_summary(vehicles_df, "OG DataFrame"))

<OG DataFrame> Missing Data Breakdown

              Missing Count  Percentage (%)
size                 306361            71.8
cylinders            177678            41.6
condition            174104            40.8
VIN                  161042            37.7
drive                130567            30.6
paint_color          130203            30.5
type                  92858            21.8
manufacturer          17646             4.1
title_status           8242             1.9
model                  5277             1.2
odometer               4400             1.0
fuel                   3013             0.7
transmission           2556             0.6
year                   1205             0.3
TOTAL               1215152           284.6


### Feature Selection and Column Pruning

To reduce dimensionality and eliminate uninformative noise, the following features are dropped prior to modeling:

* **`VIN` (Unique Identifier):** Vehicle Identification Numbers have near 1:1 cardinality with dataset rows. They carry zero generalizable predictive signal for linear regression and introduce 37.7% missingness. We rely on standard dataframe indexing instead.
* **`size` (High Sparsity & Proxy Redundancy):** Over 71.8% of the data in this column is missing. Furthermore, vehicle dimensions are effectively captured by the `type` feature (such as sedan, pickup, or SUV) and specific `model` indicators, making aggressive synthetic imputation unnecessary.
* **`paint_color` (Negligible Mass-Market Valuation Impact):** Color choice accounts for marginal price variance in standard used vehicle transactions while introducing 30.5% missingness. Excluding it prevents overparameterization across nominal categories.


In [ ]:
# Begin construction of cleaned dataframe
df = vehicles_df.copy() # standard naming of starting DataFrame
df.drop(['VIN', 'size', 'paint_color'], axis=1, inplace=True)

### Dropping Problematic Rows: `year` and `odometer`

Is there a possibility of removing any other columns? I don't think so, but I do think there is scope for removing **pieces** of other columns—specifically `year` and `title_status`.

Anyone with real-life experience buying and selling cars knows that `year` directly drives price and is very important. However, it is difficult to impute a value. As a car fan, I know it is technically possible to guess a year range from a combination of other columns (and as we'll see later, the `model` column actually includes the year for some vehicles). But building a custom filter or regex imputation function would be overly complicated, especially since other columns have missing values too. In fact, `year` is needed to help impute those other columns, which creates a circular dependency problem.

Since missing `year` values account for only **0.3%** of the dataset, the cleanest choice is to simply drop the records missing a year.

For the exact same reason, I am dropping missing values in `odometer`—another critical driver of used car prices.


In [ ]:
df.dropna(subset=['year', 'odometer'], inplace=True)

# Validation
print(
    f"NaN Counts: Year: {df['year'].isna().sum()},"
    f" Odometer: {df['odometer'].isna().sum()}"
)


NaN Counts: Year: 0, Odometer: 0


This looks much better—there are no more `NaN` values. Following ML best practices, `vehicle_age` is typically preferred over raw `year`.

### Transforming `year` to `vehicle_age`

While both metrics carry the exact same mathematical correlation with price, vehicle age directly captures annual depreciation—which is how dealerships evaluate inventory. It grounds our baseline price at a brand-new car (age 0) and prevents model drift as new inventory arrives over time.

I am transforming this data dynamically using the current calendar year via a lookup so this pipeline can run anytime without hardcoded dates.

In [ ]:
current_year = datetime.datetime.now().year
df['vehicle_age'] = current_year - df['year']

# verify a sample to ensure the years are correct
df[['year', 'vehicle_age']].sample(5)


,year,vehicle_age
184906,2013.0,13.0
268222,2013.0,13.0
45668,2013.0,13.0
369637,2003.0,23.0
140998,2003.0,23.0


The age is lining up. We can now drop the original `year`.

In [ ]:
df.drop(columns=['year'], inplace=True)

Turning my attention to the `title_status` column now:

In [ ]:
print(df['title_status'].unique())

['clean' 'rebuilt' 'lien' nan 'salvage' 'missing' 'parts only']


### Handling `title_status`

I have filtered out `NaN` and `parts only` records (<2% of data) to keep the model focused strictly on dealership-grade inventory.

Retaining `salvage`, `rebuilt`, `lien`, and `clean` titles will allow the model to accurately calculate the price penalties of title defects. In other words, it will provide valuable information on the negative correlations in this column as well.

In [ ]:
# This automatically drops 'missing', 'parts only', and 'nan'
valid_titles = ['clean', 'rebuilt', 'lien', 'salvage']
df = df[df['title_status'].isin(valid_titles)]

Now examining the `condition` colum:

In [ ]:
print(df['condition'].unique())

['good' 'excellent' 'fair' nan 'like new' 'new' 'salvage']


In [ ]:
print(get_nan_summary(df, "Cleaned DF: VIN,paint, size, title, year, odometer"))

<Cleaned DF: VIN,paint, size, title, year, odometer> Missing Data Breakdown

              Missing Count  Percentage (%)
cylinders            173170            41.9
condition            168249            40.7
drive                124706            30.2
type                  90775            22.0
manufacturer          15898             3.8
model                  5063             1.2
transmission           1643             0.4
fuel                   1623             0.4
TOTAL                581127           140.6


### Handling Missing `condition` Data
Domain & Statistical Rationale:
* Empirical Driver of Value: Vehicle condition is an established driver of market valuation (e.g., standard automotive indices like Kelley Blue Book).
* Avoiding Synthetic Distortion: Because condition is subjective and missing in ~41% of records, imputing values using other features or filling via frequency ratios risks injecting artificial bias and degrading feature correlations.
* Preserving Real-World Generalization: Creating an explicit 'Unknown' category enables the linear model to quantify the baseline value of unspecified listings directly, allowing the dealership to price incoming inventory even when physical condition ratings are omitted.

In [ ]:
df['condition'] = df['condition'].fillna('Unknown')

### Domain-Driven Imputation: `manufacturer`, `model`, and `vehicle_age`

As a car enthusiast, my domain knowledge tells me that a vehicle's core specifications are tightly determined by its **make, model, and year**.

* For example, knowing a car is a 2018 Ford Mustang gives us a very accurate proxy for its drivetrain (`rwd`), engine options, and body type (`coupe`/`convertible`).

Rather than guessing or filling missing categorical values with global averages, I am using a **hierarchical conditional imputation** strategy across our remaining problematic columns (`drive`, `cylinders`, and `type`).

I am restricting conditional imputation to two precise tiers:
1. `manufacturer` + `model` + `vehicle_age` (Generation Mode)
2. `manufacturer` + `model` (Model-level Mode)

Any edge cases that cannot be resolved at the model level are assigned to an explicit `'Unknown'` category.

Before proceeding with the imputation, the data must be examined. This will show any patterns to resolve the imputation. If nothing else, it can also be used as a baseline to see what happens afterward.

In [ ]:
# let's examine the columns before any transformation

inspect_cols = ['cylinders', 'type', 'transmission', 'fuel']

for col in inspect_cols:
  print(f"\n--- Column: {col} ---")
  print("Unique values:")
  print(df[col].unique())
  print("\nValue Counts (including NaN):")
  print(df[col].value_counts(dropna=False).head(10))


--- Column: cylinders ---
Unique values:
['8 cylinders' '6 cylinders' nan '4 cylinders' '5 cylinders' 'other'
 '3 cylinders' '10 cylinders' '12 cylinders']

Value Counts (including NaN):
cylinders
NaN             173170
6 cylinders      90550
4 cylinders      74199
8 cylinders      69990
5 cylinders       1677
10 cylinders      1415
other             1158
3 cylinders        618
12 cylinders       197
Name: count, dtype: int64

--- Column: type ---
Unique values:
['pickup' 'truck' 'other' nan 'coupe' 'SUV' 'hatchback' 'mini-van' 'sedan'
 'offroad' 'bus' 'van' 'convertible' 'wagon']

Value Counts (including NaN):
type
NaN          90775
sedan        83982
SUV          73744
pickup       42390
truck        34160
other        21293
coupe        18679
hatchback    16274
wagon        10415
van           7872
Name: count, dtype: int64

--- Column: transmission ---
Unique values:
['other' 'automatic' 'manual' nan]

Value Counts (including NaN):
transmission
automatic    325056
other         6

### Standardizing and Imputing `type`, `cylinders`, `transmission`, and `fuel`

Inspecting the unique value distributions revealed key opportunities for data refinement:

* **Category Consolidation (`type`):** `pickup` and `truck` describe the same functional vehicle class in used car markets. Combining `pickup` into `truck` prevents redundant feature splitting.
* **Extracting Numeric Cylinders:** Because cylinder counts represent physical engine size, I extracted numeric integers (`4`, `6`, `8`) to allow the linear model to capture linear scale rather than treating each engine format as an isolated nominal category.

I can proceed with the Missing values across `type`, `cylinders`, `transmission`, and `fuel` through the hierarchical imputation suggested above: `make + model + age` $\rightarrow$ `make + model` $\rightarrow$ `make`.


Unfortunatelly, looking at our `NaN` values above, `manufacturer` and `model` themselves have some empty spots. I wanted to examine this first.

In [ ]:
target_indices = [384, 2241, 2826, 3420]

# Pull exact target rows
target_rows = df.loc[target_indices, ['manufacturer', 'model']]

# Pull reproducible random sample for the rest
random_sample = (
    df.drop(index=target_indices)[['manufacturer', 'model']]
    .sample(10, random_state=42)
)

# Display combined sample
final_sample = pd.concat([target_rows, random_sample])
print("Target Edge Cases + Random Sample:")
print(final_sample)

Target Edge Cases + Random Sample:
       manufacturer                       model
384             bmw                          x6
2241           ford                 f150 lariat
2826      chevrolet              silverado 2500
3420           ford   f150 super cab xlt pickup
320807         ford  f-250 super duty lariat li
388767         mini                    cooper s
400589          bmw                x1 xdrive28i
40598       lincoln         continental reserve
181019         ford    f350 super duty crew cab
82239          ford        super duty f-350 drw
176615         ford                      taurus
145420       toyota                  highlander
13062        toyota                         NaN
190276      hyundai                      sonata


Unfortunately `manufacturer` itself has a high percentage of missing values. Further inspection of the groups revealed that missing `manufacturer` values are often leaked directly into the `model` column.

Meanwhile, the data in `model` is highly corrupted with typos (e.g., 'yundai'), promotional noise, emojis, Cyrillic characters, and mathematical alphanumeric fonts. Let's see all values that are not alphanumeric values:



In [ ]:
non_ascii_models = df[
    df['model'].astype(str).str.contains(r'[^\x00-\x7F]', regex=True, na=False)
][['manufacturer', 'model']]

print("Non-ASCII / Stylized Models Found:")
print(non_ascii_models.head(20))

Non-ASCII / Stylized Models Found:
         manufacturer                        model
16507             NaN  🔥GMC Sierra 1500 SLE🔥 4X4 🔥
21194          nissan                     altima ￼
22997          toyota                    corolla￼￼
23007          nissan                     altima ￼
39447             NaN                         50’s
41153   mercedes-benz            c-class c 43 amg®
44302       chevrolet           12’ flatbed atruck
51933             NaN                 1937 Willy’s
57669            ford     flex sel sport – 3rd row
89715        chrysler          300 touring édition
96985          toyota                  corolla “s”
99400             NaN         𝓜𝓮𝓻𝓬𝓮𝓭𝓮𝓼 𝓫𝓮𝓷𝔃 𝓶𝓵 350
139481            NaN               VMI-CHRYSLER-♿
139801       chrysler                        ♿ vmi
140276            kia                 ​​sorento lx
142548       chrysler                    / vmi / ♿
142936       chrysler                    * vmi * ♿
143075       chrysler                    * vmi 

### Dropping Incomplete and Malformed `manufacturer`/`model` Rows

I initially considered defining increasingly complex extraction algorithms to parse missing makes and models—such as fuzzy string matching to detect known models, or creating lookup dictionaries for shorthand slang (e.g., `vette` for `corvette` or `n altima` for `nissan`).

However, looking closely at the data revealed several real-world text challenges:
* **Embedded Make and Year:** The manufacturer name was often included inside the model string rather than as the first word because sellers listed the manufacturing year first.
* **Merged Brand Strings:** Some listings had the make and model clubbed together without spaces.
* **Unicode and Emojis:** Characters like stylized script (`𝓜𝓮𝓻𝓬𝓮𝓭𝓮𝓼`) and emojis (`🔥`, `♿`) were scattered across listings.
* **Garbage and Raw Identifier Data:** Several entries contained raw VIN strings or blank fields rather than actual vehicle names, such as:
  * `('5fnyf4h21eb053247', '')`
  * `('5n1aa0ne3bn614984', '')`
  * `('5nmsh73e27h004000', '')`

Here I was forced to take a step back. In our current dataset, missing `manufacturer` data accounts for **3.8%** (15,898 records) while missing `model` accounts for **1.2%** (5,063 records). Combined with malformed entries, they represent only ~5% of our rows.

Attempting to build and maintain a custom string-matching extraction algorithm is highly inefficient and risks injecting heuristic errors. Dropping these missing, malformed, and non-ASCII rows leaves over 400,000 statistically clean records—giving us a robust dataset without introducing synthetic data noise.

Therefore, I am dropping records missing a `manufacturer` or `model`. As a final safeguard, I sanitize any non-ASCII characters, emojis, and unicode artifacts across the remaining text to keep our categories clean.

In [ ]:
print(f"Before drop: {len(df):,} rows")

# 1. Drop rows with missing manufacturer or model
df = df.dropna(subset=["manufacturer", "model"]).reset_index(drop=True)

# 2. Filter out non-ASCII characters (emojis, math script fonts, etc.)
ascii_mask = (
    df["model"].astype(str).str.contains(r"^[ -~]+$", regex=True, na=False)
    & df["manufacturer"]
    .astype(str)
    .str.contains(r"^[ -~]+$", regex=True, na=False)
)
df = df[ascii_mask].reset_index(drop=True)

# 3. Clean casing and whitespace
df["manufacturer"] = df["manufacturer"].astype(str).str.lower().str.strip()
df["model"] = df["model"].astype(str).str.lower().str.strip()

print(f"After drop: {len(df):,} rows")


Before drop: 412,974 rows
After drop: 391,963 rows


Validating:

In [ ]:
non_ascii_models = df[
    df['model'].astype(str).str.contains(r'[^\x00-\x7F]', regex=True, na=False)
    | df['manufacturer']
    .astype(str)
    .str.contains(r'[^\x00-\x7F]', regex=True, na=False)
][['manufacturer', 'model']]

print('Non-ASCII / Stylized Models Found:')
print(non_ascii_models.head(20))


Non-ASCII / Stylized Models Found:
Empty DataFrame
Columns: [manufacturer, model]
Index: []


Our text is free of non-ascii. As an additional baseline:

In [ ]:
pre_imputation_df = get_nan_df(df)

print(get_nan_summary(
    df, 'Cleaned DF: Pre-Imputation Snapshot'
))

<Cleaned DF: Pre-Imputation Snapshot> Missing Data Breakdown

              Missing Count  Percentage (%)
cylinders            162945            41.6
drive                115714            29.5
type                  82503            21.0
fuel                   1566             0.4
transmission           1556             0.4
TOTAL                364284            92.9


### Next Step: Hierarchical Imputation

With `manufacturer`, `model`, and `vehicle_age` now fully validated and clean, I am proceeding with my multi-tiered hierarchical mode imputation to fill the remaining missing values across `cylinders`, `drive`, `type`, `fuel`, and `transmission`.


In [ ]:
# 1. Snapshot NaN counts before imputation for verification
impute_target_cols = [
    'drive',
    'cylinders',
    'type',
    'transmission',
    'fuel',
]
pre_impute_nans = df[impute_target_cols].isna().sum()

# 2. Consolidate 'pickup' into 'truck' in body type before mode calculations
df['type'] = df['type'].replace({'pickup': 'truck'})


# Helper function to compute mode Series safely
def get_mode_series(group_series: pd.Series) -> pd.Series:
  valid = group_series.dropna()
  return (
      group_series.fillna(valid.mode()[0])
      if not valid.empty and not valid.mode().empty
      else group_series
  )


# 3. Hierarchical Mode Imputation (Make+Model+Age -> Make+Model)
structural_cols = ['drive', 'cylinders', 'type', 'transmission', 'fuel']

for col in structural_cols:
  # 3.1: Most granular (Make + Model + Age)
  m1 = df.groupby(['manufacturer', 'model', 'vehicle_age'])[col].transform(
      get_mode_series
  )

  # 3.2: Make + Model
  m2 = df.groupby(['manufacturer', 'model'])[col].transform(get_mode_series)

  # 3.3: Apply hierarchically back to the column
  df[col] = df[col].fillna(m1).fillna(m2)

In [ ]:
post_imputation_df = get_nan_df(df)

print(get_nan_summary(
    df, 'Cleaned DF: Post-Imputation Snapshot'
))

pre = pre_imputation_df.drop(index='TOTAL', errors='ignore')
post = post_imputation_df.drop(index='TOTAL', errors='ignore')

# 1. Align all column names that had missing values before imputation
problem_cols = pre[pre['Missing Count'] > 0].index

# 2. Extract counts; if absent in post, count is 0 (100% recovered)
before_counts = pre.loc[problem_cols, 'Missing Count'].astype(int)
after_counts = (
    post['Missing Count'].reindex(problem_cols).fillna(0).astype(int)
)

# 3. Calculate Delta (number of recovered missing cells) and Recovery %
delta = before_counts - after_counts
recovery_pct = (
    (delta / before_counts * 100).fillna(100.0).round(1).astype(str) + '%'
)

comparison_df = pd.DataFrame(
    {
        f'Pre Count': before_counts,
        f'Post Count': after_counts,
        'Δ Recovered': delta,
        'Recovery (%)': recovery_pct,
    },
    index=problem_cols,
).sort_values(by='Δ Recovered', ascending=False)

# 4. Add summary TOTAL row
total_before = before_counts.sum()
total_after = after_counts.sum()
total_delta = total_before - total_after
total_recovery_pct = (
    f'{round((total_delta / total_before * 100), 1) if total_before > 0 else 100.0}%'
)

comparison_df.loc['TOTAL'] = [
    total_before,
    total_after,
    total_delta,
    total_recovery_pct,
]

print(comparison_df)

<Cleaned DF: Post-Imputation Snapshot> Missing Data Breakdown

              Missing Count  Percentage (%)
cylinders             33289             8.5
drive                 18017             4.6
type                   6455             1.6
fuel                   1024             0.3
transmission             62             0.0
TOTAL                 58847            15.0
              Pre Count  Post Count  Δ Recovered Recovery (%)
cylinders        162945       33289       129656        79.6%
drive            115714       18017        97697        84.4%
type              82503        6455        76048        92.2%
transmission       1556          62         1494        96.0%
fuel               1566        1024          542        34.6%
TOTAL            364284       58847       305437        83.8%


### Hierarchical Imputation Results & Assigning `'Unknown'` Sentinels

The hierarchical conditional imputation successfully recovered **305,437 missing values** across my dataset, achieving an overall **83.8% recovery rate** based on make, model, and age groupings. Specifically, `drive` recovered **84.4%** (97,697 records) and `cylinders` recovered **79.6%** (129,656 records), mapping them cleanly back to their true mechanical specifications.

For the remaining ~16% of missing categorical entries, I am opting to fill them with an explicit `'Unknown'` sentinel rather than dropping rows or forcing synthetic guesses:

* **Preventing Sample Attrition:** Dropping rows with any remaining missing field would discard tens of thousands of otherwise complete vehicle records, reducing statistical power.
* **Avoiding Fabricated Variance:** When an obscure or vintage vehicle lacks a clear mode in its cohort, guessing a category injects artificial assumptions.
* **Real-World Inference (Out-of-Distribution Handling):** In production, dealerships frequently encounter trade-in listings with omitted attributes. Giving the linear regression model an explicit `'Unknown'` category allows it to learn an unpenalized baseline coefficient for missing entries.


In [ ]:
# Fill remaining NaNs across all categorical features with 'Unknown'
impute_cols = [
    'drive',
    'cylinders',
    'type',
    'transmission',
    'fuel',
]
df[impute_cols] = df[impute_cols].fillna('Unknown')

# Confirm all NaNs in categorical columns are resolved
print(get_nan_summary(df, 'Final Cleaned DF: Post Imputation+Unknown Fill'))

<Final Cleaned DF: Post Imputation+Unknown Fill>: No missing values detected.

Empty DataFrame
Columns: [Missing Count, Percentage (%)]
Index: []


In [ ]:
unknown_cols = impute_cols + ['condition']

for col in unknown_cols:
  print(f"\n--- Column: {col} ---")
  print("Unique values:")
  print(df[col].unique())
  print("\nValue Counts (including NaN):")
  print(df[col].value_counts(dropna=False).head(10))


--- Column: drive ---
Unique values:
['4wd' 'rwd' 'Unknown' 'fwd']

Value Counts (including NaN):
drive
4wd        168136
fwd        136378
rwd         69432
Unknown     18017
Name: count, dtype: int64

--- Column: cylinders ---
Unique values:
['8 cylinders' '6 cylinders' 'Unknown' '4 cylinders' '5 cylinders'
 '3 cylinders' '10 cylinders' 'other' '12 cylinders']

Value Counts (including NaN):
cylinders
6 cylinders     133031
4 cylinders     121021
8 cylinders      98105
Unknown          33289
5 cylinders       2220
10 cylinders      1804
other             1532
3 cylinders        830
12 cylinders       131
Name: count, dtype: int64

--- Column: type ---
Unique values:
['truck' 'other' 'SUV' 'coupe' 'hatchback' 'mini-van' 'sedan' 'offroad'
 'van' 'convertible' 'Unknown' 'wagon' 'bus']

Value Counts (including NaN):
type
sedan          102703
SUV             93740
truck           92181
other           20828
coupe           20578
hatchback       17490
wagon           11831
van            

### Consolidating `'Unknown'` and `'other'` Categories

Examining the category distributions shows that both `'other'` and `'Unknown'` represent unclassified or non-standard listings. In columns like `transmission` (62 instances) and `fuel` (1,024 instances), retaining `'Unknown'` as an isolated category introduces unnecessary sparsity without providing distinct predictive value.

To keep the feature space parsimonious and prevent the linear model from fitting redundant coefficients, I am consolidating all `'Unknown'` labels into the existing `'other'` category across `transmission`, `fuel`, `type`, and `drive`.


In [ ]:
for col in unknown_cols:
  if col in df.columns:
    df[col] = df[col].replace({'Unknown': 'other', 'unknown': 'other'})
    # Ensure all strings are lowercase
    df[col] = df[col].astype(str).str.lower()

# Verification check
for col in unknown_cols:
  print(f"\n--- Consolidated Counts for {col} ---")
  print(df[col].value_counts())


--- Consolidated Counts for drive ---
drive
4wd      168136
fwd      136378
rwd       69432
other     18017
Name: count, dtype: int64

--- Consolidated Counts for cylinders ---
cylinders
6 cylinders     133031
4 cylinders     121021
8 cylinders      98105
other            34821
5 cylinders       2220
10 cylinders      1804
3 cylinders        830
12 cylinders       131
Name: count, dtype: int64

--- Consolidated Counts for type ---
type
sedan          102703
suv             93740
truck           92181
other           27283
coupe           20578
hatchback       17490
wagon           11831
van              9853
convertible      8176
mini-van         6937
offroad           677
bus               514
Name: count, dtype: int64

--- Consolidated Counts for transmission ---
transmission
automatic    311674
other         59433
manual        20856
Name: count, dtype: int64

--- Consolidated Counts for fuel ---
fuel
gas         330762
other        29559
diesel       25079
hybrid        4985
elect

In [ ]:
print(vehicles_df.info())
print(get_nan_summary(vehicles_df, 'OG DataFrame'))

print('='*22+'+++'+'='*22)

print(df.info())
print(get_nan_summary(df, 'Cleaned and Imputed DataFrame'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   region        426880 non-null  object 
 2   price         426880 non-null  int64  
 3   year          425675 non-null  float64
 4   manufacturer  409234 non-null  object 
 5   model         421603 non-null  object 
 6   condition     252776 non-null  object 
 7   cylinders     249202 non-null  object 
 8   fuel          423867 non-null  object 
 9   odometer      422480 non-null  float64
 10  title_status  418638 non-null  object 
 11  transmission  424324 non-null  object 
 12  VIN           265838 non-null  object 
 13  drive         296313 non-null  object 
 14  size          120519 non-null  object 
 15  type          334022 non-null  object 
 16  paint_color   296677 non-null  object 
 17  state         426880 non-null  object 
dtypes: f

## Summary: Data Understanding & Cleanup

By removing uninformative identifiers (`VIN`, `size`, `paint_color`), pruning extreme noise (<2% of titles and missing years), and dropping unrecoverable makes/models, I reduce the dataset from **426,880 rows down to 391,963 clean records** (a net retention of 91.8%).

Through hierarchical domain imputation, all 1.2+ million initial missing instances across `drive`, `cylinders`, `type`, `transmission`, and `fuel` are completely resolved to **0 missing values**, with subjective missing condition entries safely captured in an explicit `other`/`Unknown` category (160,101 rows).

With a fully populated and statistically clean dataset of 391,963 records across 15 structured features, I am ready to proceed to the next phase of EDA.


### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`.

In [ ]:
# store a copy of the cleaned df just in case
cleaned_df = df.copy()

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.